# Geometry of Social Identity × Location Representations

**Author:** Mostafa El-Kalliny, building on analysis code by Will Sheeran

**Note:** Analysis code written with AI assistance (Claude) and manually reviewed and validated by M.E.K.

## Imports

In [ ]:
%matplotlib widget

from scipy import stats
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from scipy.spatial.distance import euclidean
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

## Parameters

In [ ]:
P2P_MODE   = 'normalized'
METRIC     = 'accuracy'   # 'accuracy', 'f1', or 'auroc'

PICKLE_DIR   = os.path.expanduser('~/Documents/lab/vca2/session_pickles')
RESULTS_DIR  = os.path.expanduser('~/Documents/lab/vca2/ccgp_results')
# ── Session-exclusion toggle ─────────────────────────────────────────────────
EXCLUDE_MODE = True   # Set True to drop the sessions listed below, per consistency with other analyses

# Each entry is (animal_id_str, session_idx) where session_idx follows
# SESSION_LABELS 1-based order: 1=SS_NN, 2=SS_NF, 3=OS_NN, 4=OS_NF, 5=OS_NFS
EXCLUDED_SESSION_LIST = [
    ('8020', 5),
    ('7981', 3),
    ('7981', 4),
    ('7845', 4),
    ('6175', 5),
]

_suffix     = '_excluded' if EXCLUDE_MODE else ''
OUTPUT_DIR   = os.path.expanduser(f'~/Documents/lab/vca2/figures/pca_geometry{_suffix}')

IDENTITY_COLOR = '#A23B72'
LOCATION_COLOR = '#2E86AB'

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

SESSION_LABELS = ['SS_NN', 'SS_NF', 'OS_NN', 'OS_NF', 'OS_NFS']

from itertools import combinations
ALL_PAIRWISE_COMPARISONS = [
    (sl_a, sl_b, f'{sl_a} vs {sl_b}')
    for sl_a, sl_b in combinations(SESSION_LABELS, 2)
]

SESSION_TITLES = {
    'SS_NN':  'Day 1: Same-Sex, Novel vs. Novel',
    'SS_NF':  'Day 2: Same-Sex, Novel vs. Familiar',
    'OS_NN':  'Day 3: Opposite-Sex, Novel vs. Novel',
    'OS_NF':  'Day 4: Opposite-Sex, Novel vs. Familiar',
    'OS_NFS': 'Day 5: Opposite-Sex, Familiar vs. Novel (Post-Separation)',
}

SESSION_XTICK_LABELS = ['SS-N', 'SS-F', 'OS-N', 'OS-F\nPre', 'OS-F\nPost']

CONDITION_COLORS = {
    'pr': '#4B2E8D',
    'pl': '#9382BB',
    'nr': '#07AC4B',
    'nl': '#6ACD93',
}
CONDITION_LABELS = {
    'pr': 'Partner / Right',
    'pl': 'Partner / Left',
    'nr': 'Novel / Right',
    'nl': 'Novel / Left',
}

CONDITIONS = ['pr', 'pl', 'nr', 'nl']

EDGES = [
    ('pr', 'pl'),
    ('nr', 'nl'),
    ('pr', 'nr'),
    ('pl', 'nl'),
]

# Colors for the three distance axes in the line plot
AXIS_COLORS = {
    'identity': '#C0392B',   # red
    'location': '#2980B9',   # blue
    'diagonal': '#7F8C8D',   # gray
}

AXIS_LABELS = {
    'identity': 'Identity axis (partner vs novel)',
    'location': 'Location axis (left vs right)',
    'diagonal': 'Diagonal (XOR pairs)',
}

## Helper Functions

In [ ]:
def compute_condition_centroid(pt_pvs_dict, condition):
    """
    Average all pseudotrial pvs for one condition -> one centroid vector.
    """
    pvs = pt_pvs_dict[condition]
    matrix = np.stack([v.values for v in pvs.values()])  # (n_pseudotrials x numcells)
    return matrix.mean(axis=0)


def compute_bout_averages(pt_pvs_dict, pseudotrials_dict, condition):
    """
    For one condition, compute one averaged population vector per bout
    by grouping pseudotrials by their parent bout (the 'int' column).

    Returns np.ndarray of shape (n_bouts, numcells).
    """
    pvs = pt_pvs_dict[condition]
    pts = pseudotrials_dict[condition]

    bout_vectors = []
    for bout_id, group in pts.groupby('int'):
        pt_indices = group.index.tolist()
        bout_matrix = np.stack([pvs[i].values for i in pt_indices if i in pvs])
        bout_vectors.append(bout_matrix.mean(axis=0))

    return np.stack(bout_vectors)


def build_animal_centroids(animal_data, conditions=None):
    if conditions is None:
        conditions = CONDITIONS
    return np.stack([
        compute_condition_centroid(animal_data['pt_pvs'], cond)
        for cond in conditions
    ])

def build_pseudopopulation_centroids(session_data):
    """
    Build 4 condition centroids in pseudopopulation space.
    Returns centroids (4, N_total) and list of contributing animal keys.
    """
    included = [k for k, v in session_data.items() if v.get('included')]
    if len(included) == 0:
        raise ValueError('No included animals in this session')

    condition_vectors = {cond: [] for cond in CONDITIONS}
    valid_animals = []
    skipped = []

    for key in included:
        try:
            for cond in CONDITIONS:
                centroid = compute_condition_centroid(session_data[key]['pt_pvs'], cond)
                condition_vectors[cond].append(centroid)
            valid_animals.append(key)
        except Exception as e:
            skipped.append((key, str(e)))

    if skipped:
        print('  Skipped during centroid computation:')
        for k, reason in skipped:
            print(f'    {k}: {reason}')

    centroids = np.stack([
        np.concatenate(condition_vectors[cond]) for cond in CONDITIONS
    ])

    return centroids, valid_animals


def run_pca_on_centroids(centroids):
    """
    Run PCA on a 4 x N centroid matrix.
    Returns coords (4, 3), fitted pca object, variance_explained (3,).
    """
    pca = PCA(n_components=3)
    coords = pca.fit_transform(centroids)
    return coords, pca, pca.explained_variance_ratio_


def compute_centroid_distances(centroids):
    """
    Compute pairwise Euclidean distances between the 4 condition centroids
    in the full neural space, collapsed into three axis types.

    Parameters:
    -----------
    centroids : np.ndarray (4, numcells)
        Row order = CONDITIONS = ['pr', 'pl', 'nr', 'nl']

    Returns:
    --------
    distances : dict with keys 'identity', 'location', 'diagonal'
        Each value is the mean of the two relevant pairwise distances
    raw : dict with all six individual pairwise distances
    """
    idx = {cond: i for i, cond in enumerate(CONDITIONS)}

    def dist(c1, c2):
        return euclidean(centroids[idx[c1]], centroids[idx[c2]])

    raw = {
        'pr_pl': dist('pr', 'pl'),
        'nr_nl': dist('nr', 'nl'),
        'pr_nr': dist('pr', 'nr'),
        'pl_nl': dist('pl', 'nl'),
        'pr_nl': dist('pr', 'nl'),
        'pl_nr': dist('pl', 'nr'),
    }

    distances = {
        'identity': np.mean([raw['pr_pl'], raw['nr_nl']]),
        'location': np.mean([raw['pr_nr'], raw['pl_nl']]),
        'diagonal': np.mean([raw['pr_nl'], raw['pl_nr']]),
    }

    return distances, raw


def plot_geometry_3d_per_animal(coords, pca, bout_vectors_by_cond,
                                variance_explained, title, save_path=None):
    """
    Per-animal plot: bout averages as small points, centroids as large markers.
    """
    coord_dict = {cond: coords[i] for i, cond in enumerate(CONDITIONS)}

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Bout points
    for cond in CONDITIONS:
        bout_vecs = bout_vectors_by_cond[cond]
        if len(bout_vecs) == 0:
            continue
        bout_coords = pca.transform(bout_vecs)
        ax.scatter(
            bout_coords[:, 0], bout_coords[:, 1], bout_coords[:, 2],
            color=CONDITION_COLORS[cond],
            s=30, alpha=0.4, zorder=2, edgecolors='none'
        )

    # Edges
    for c1, c2 in EDGES:
        p1, p2 = coord_dict[c1], coord_dict[c2]
        ax.plot(
            [p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
            color='gray', linewidth=1.2, alpha=0.5, zorder=3
        )

    # Centroids
    for cond in CONDITIONS:
        x, y, z = coord_dict[cond]
        ax.scatter(x, y, z,
                   color=CONDITION_COLORS[cond], s=150, zorder=5,
                   label=CONDITION_LABELS[cond],
                   edgecolors='white', linewidths=1.0)
        ax.text(x, y, z, f'  {cond}', fontsize=9, color=CONDITION_COLORS[cond])

    ax.set_xlabel(f'PC1 ({variance_explained[0]*100:.1f}%)', fontsize=9, labelpad=8)
    ax.set_ylabel(f'PC2 ({variance_explained[1]*100:.1f}%)', fontsize=9, labelpad=8)
    ax.set_zlabel(f'PC3 ({variance_explained[2]*100:.1f}%)', fontsize=9, labelpad=8)
    ax.set_title(title, fontsize=9, pad=12)
    ax.legend(loc='upper left', fontsize=7, framealpha=0.7)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'    Saved: {save_path}')

    plt.show()

def is_excluded(animal_key, session_label):
    """
    Return True if this (animal_key, session_label) pair should be dropped
    when EXCLUDE_MODE is active.  animal_key is the full pickle key, e.g.
    '8020_20250403_SS_NN'; animal_id is the leading numeric string.
    """
    if not EXCLUDE_MODE:
        return False
    animal_id   = animal_key.split('_')[0]
    session_idx = SESSION_LABELS.index(session_label) + 1
    return (animal_id, session_idx) in set(EXCLUDED_SESSION_LIST)


def filter_session_data(session_data, session_label):
    """
    Return a copy of session_data with excluded animals removed
    (their 'included' flag set to False, so all downstream logic is unchanged).
    """
    if not EXCLUDE_MODE:
        return session_data
    filtered = {}
    for k, v in session_data.items():
        if is_excluded(k, session_label) and v.get('included', False):
            v = dict(v)   # shallow copy so we don't mutate the pickle
            v['included']        = False
            v['exclusion_reason'] = 'manually excluded via EXCLUDED_SESSION_LIST'
        filtered[k] = v
    return filtered



def run_lmm(df, outcome_col, group_col, session_col, animal_col):
    """
    Generic LMM
    outcome_col: column name for the dependent variable
    group_col: column name for the main fixed effect (e.g. 'condition' or 'decode_type')
    """
    df1 = df.copy()

    # Encode session as integer string categories
    df1[session_col] = pd.Categorical(
        df1[session_col].map({sl: str(i+1) for i, sl in enumerate(SESSION_LABELS)}),
        categories=['1', '2', '3', '4', '5']
    )
    assert df1[session_col].isna().sum() == 0, \
        f"NaN in {session_col} after categorical conversion"

    m = smf.mixedlm(
        f"{outcome_col} ~ {group_col} * {session_col}",
        data=df1,
        groups=df1[animal_col]
    )
    result = m.fit()
    print(result.summary())

    # Joint F-tests
    interaction_names  = [n for n in result.fe_params.index if ':' in n]
    session_main_names = [n for n in result.fe_params.index
                          if session_col in n and ':' not in n]
    group_main_names   = [n for n in result.fe_params.index
                          if group_col in n and ':' not in n]

    f_test_group = result.f_test(group_main_names)
    f_test_sesh  = result.f_test(session_main_names)
    f_test_int   = result.f_test(interaction_names)

    print("\n=== Joint F-tests ===")
    print(f"{group_col} main effect:  F({f_test_group.df_num:.0f},{f_test_group.df_denom:.0f}) = {f_test_group.fvalue:.3f},  p = {f_test_group.pvalue:.4f}")
    print(f"Session main effect:     F({f_test_sesh.df_num:.0f},{f_test_sesh.df_denom:.0f}) = {f_test_sesh.fvalue:.3f},  p = {f_test_sesh.pvalue:.4f}")
    print(f"Interaction:             F({f_test_int.df_num:.0f},{f_test_int.df_denom:.0f}) = {f_test_int.fvalue:.3f},  p = {f_test_int.pvalue:.4f}")

    # Get main effect coefficient
    group_coef_name = [n for n in result.fe_params.index
                       if group_col in n and ':' not in n and '[T.' in n][0]

    results_dict = {
        'group_col':              group_col,
        'group_coef_name':        group_coef_name,
        'group_coef':             result.fe_params[group_coef_name],
        'group_coef_se':          result.bse_fe[group_coef_name],
        'group_coef_ci_low':      result.conf_int().loc[group_coef_name, 0],
        'group_coef_ci_high':     result.conf_int().loc[group_coef_name, 1],
        'group_main_F':           f_test_group.fvalue,
        'group_main_p':           f_test_group.pvalue,
        'group_main_df_num':      f_test_group.df_num,
        'group_main_df_denom':    f_test_group.df_denom,
        'session_main_F':         f_test_sesh.fvalue,
        'session_main_p':         f_test_sesh.pvalue,
        'session_main_df_num':    f_test_sesh.df_num,
        'session_main_df_denom':  f_test_sesh.df_denom,
        'interaction_F':          f_test_int.fvalue,
        'interaction_p':          f_test_int.pvalue,
        'interaction_df_num':     f_test_int.df_num,
        'interaction_df_denom':   f_test_int.df_denom,
        'AIC':                    result.aic,
        'BIC':                    result.bic,
        'n_obs':                  result.nobs,
        'random_effect_var':      result.cov_re.iloc[0, 0],
        'residual_var':           result.scale,
    }

    return results_dict


## Pseudopopulation PCA

In [ ]:
# Pseudopopulation PCA

pseudopop_summary = []

for session_label in SESSION_LABELS:
    pickle_path = os.path.join(PICKLE_DIR, f'{session_label}.pkl')
    if not os.path.isfile(pickle_path):
        print(f'\n{session_label}: pickle not found -- skipping')
        continue

    print(f'\n{"="*60}')
    print(f'PSEUDOPOPULATION | {SESSION_TITLES[session_label]}')

    with open(pickle_path, 'rb') as f:
        session_data = pickle.load(f)
    session_data = filter_session_data(session_data, session_label)

    included = [k for k, v in session_data.items() if v.get('included')]
    excluded = [k for k, v in session_data.items() if not v.get('included')]
    print(f'  Included: {len(included)}  |  Excluded: {len(excluded)}')
    for k in excluded:
        print(f'    EXCLUDED {k}: {session_data[k]["exclusion_reason"]}')

    if len(included) == 0:
        print('  No included animals -- skipping')
        del session_data
        continue

    try:
        centroids, valid_animals = build_pseudopopulation_centroids(session_data)
        coords, pca, var_exp = run_pca_on_centroids(centroids)

        n_animals  = len(valid_animals)
        n_neurons  = centroids.shape[1]
        animal_ids = [a.split('_')[0] for a in valid_animals]

        print(f'  Animals: {animal_ids}')
        print(f'  Total neurons: {n_neurons}')
        print(f'  PC1: {var_exp[0]*100:.1f}%  PC2: {var_exp[1]*100:.1f}%  '
              f'PC3: {var_exp[2]*100:.1f}%  Total: {var_exp.sum()*100:.1f}%')

        # ── CSV ──────────────────────────────────────────────────────────────
        rows = []
        for i, cond in enumerate(CONDITIONS):
            rows.append({
                'session':        session_label,
                'session_title':  SESSION_TITLES[session_label],
                'n_animals':      n_animals,
                'n_neurons':      n_neurons,
                'animals':        ','.join(animal_ids),
                'condition':      cond,
                'pc1':            coords[i, 0],
                'pc2':            coords[i, 1],
                'pc3':            coords[i, 2],
                'pc1_var_exp':    var_exp[0],
                'pc2_var_exp':    var_exp[1],
                'pc3_var_exp':    var_exp[2],
            })

        csv_path = os.path.join(OUTPUT_DIR, f'{session_label}_pseudopop_pca.csv')
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f'  Saved CSV: {csv_path}')

        # ── Plot ─────────────────────────────────────────────────────────────
        coord_dict = {cond: coords[i] for i, cond in enumerate(CONDITIONS)}

        fig = plt.figure(figsize=(7, 6))
        ax  = fig.add_subplot(111, projection='3d')

        # Clean pane faces — distinct shading for depth perception
        ax.xaxis.pane.fill = True
        ax.yaxis.pane.fill = True
        ax.zaxis.pane.fill = True
        ax.xaxis.pane.set_facecolor('#E8E8E8')
        ax.yaxis.pane.set_facecolor('#D8D8D8')
        ax.zaxis.pane.set_facecolor('#C8C8C8')
        ax.xaxis.pane.set_edgecolor('#BBBBBB')
        ax.yaxis.pane.set_edgecolor('#BBBBBB')
        ax.zaxis.pane.set_edgecolor('#BBBBBB')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.grid(False)
        ax.set_axis_on()

        for c1, c2 in EDGES:
            p1, p2 = coord_dict[c1], coord_dict[c2]
            ax.plot(
                [p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                color='gray', linewidth=1.2, alpha=0.5, zorder=1
            )

        for cond in CONDITIONS:
            x, y, z = coord_dict[cond]
            ax.scatter(x, y, z,
                       color=CONDITION_COLORS[cond], s=120, zorder=5,
                       label=CONDITION_LABELS[cond],
                       edgecolors='white', linewidths=0.8)
            ax.text(x, y, z, f'  {cond}', fontsize=11,
                    color=CONDITION_COLORS[cond], fontweight='bold')

        ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)', fontsize=9, labelpad=8)
        ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)', fontsize=9, labelpad=8)
        ax.set_zlabel(f'PC3 ({var_exp[2]*100:.1f}%)', fontsize=9, labelpad=8)

        title = (
            f'{SESSION_TITLES[session_label]}\n'
            f'Pseudopopulation | n={n_animals} animals | '
            f'PC3: {var_exp[2]*100:.1f}%'
        )
        ax.set_title(title, fontsize=9, pad=12)
        ax.legend(loc='upper left', fontsize=7, framealpha=0.7)
        plt.tight_layout()

        for ext in ('png', 'svg'):
            out = os.path.join(OUTPUT_DIR, f'{session_label}_pseudopop.{ext}')
            plt.savefig(out, dpi=200, bbox_inches='tight')
            print(f'  Saved: {out}')

        plt.show()

        pseudopop_summary.append({
            'session':   session_label,
            'n_animals': n_animals,
            'n_neurons': n_neurons,
            'pc1_var':   var_exp[0],
            'pc2_var':   var_exp[1],
            'pc3_var':   var_exp[2],
        })

    except Exception as e:
        print(f'  ERROR: {e}')

    del session_data

print(f'\n{"="*60}')
print('Pseudopopulation done.')

## Point-to-Plane Distance

In [ ]:
def point_to_plane_distance(centroids):
    raw_distances  = []
    norm_distances = []
    for i in range(4):
        others = [j for j in range(4) if j != i]
        p0     = centroids[others[0]]
        p1     = centroids[others[1]]
        p2     = centroids[others[2]]
        fourth = centroids[i]

        displacement = fourth - p0
        v1 = p1 - p0
        v2 = p2 - p0

        v1_norm  = v1 / (np.linalg.norm(v1) + 1e-12)
        v2_orth  = v2 - np.dot(v2, v1_norm) * v1_norm
        v2_norm  = v2_orth / (np.linalg.norm(v2_orth) + 1e-12)

        in_plane     = (np.dot(displacement, v1_norm) * v1_norm +
                        np.dot(displacement, v2_norm) * v2_norm)
        out_of_plane = displacement - in_plane
        raw_dist     = np.linalg.norm(out_of_plane)
        raw_distances.append(raw_dist)

        among_three = np.mean([
            euclidean(centroids[others[0]], centroids[others[1]]),
            euclidean(centroids[others[0]], centroids[others[2]]),
            euclidean(centroids[others[1]], centroids[others[2]]),
        ])
        norm_distances.append(raw_dist / (among_three + 1e-12))

    return np.mean(raw_distances), np.mean(norm_distances)


def get_p2p_val(raw, norm):
    return raw if P2P_MODE == 'raw' else norm


# Collect
p2p_records = []

for session_label in SESSION_LABELS:
    pickle_path = os.path.join(PICKLE_DIR, f'{session_label}.pkl')
    if not os.path.isfile(pickle_path):
        continue
    with open(pickle_path, 'rb') as f:
        session_data = pickle.load(f)
    session_data = filter_session_data(session_data, session_label)
    included = [k for k, v in session_data.items() if v.get('included')]
    for animal_key in included:
        animal_data = session_data[animal_key]
        animal_id   = animal_key.split('_')[0]
        try:
            centroids         = build_animal_centroids(animal_data)
            raw_p2p, norm_p2p = point_to_plane_distance(centroids)
            p2p_records.append({
                'animal':   animal_id,
                'session':  session_label,
                'p2p':      raw_p2p,
                'p2p_norm': norm_p2p,
                'val':      get_p2p_val(raw_p2p, norm_p2p),
            })
        except Exception as e:
            print(f'  Skipped {animal_key}: {e}')

p2p_df = pd.DataFrame(p2p_records)

# CSV
p2p_csv_path = os.path.join(OUTPUT_DIR, f'point_to_plane_social_{P2P_MODE}.csv')
p2p_df[['animal', 'session', 'p2p', 'p2p_norm']].to_csv(p2p_csv_path, index=False)
print(f'Saved CSV: {p2p_csv_path}')

print(f'\nSOCIAL p2p ({P2P_MODE}) — mean per session:')
print(p2p_df.groupby('session')[['p2p', 'p2p_norm']]
          .mean().reindex(SESSION_LABELS).round(4).to_string())

# Plot
ylabel = ('Point-to-plane distance (z-scored dF/F units)'
          if P2P_MODE == 'raw' else 'Normalized point-to-plane distance')

fig, ax = plt.subplots(figsize=(7, 5))

x = np.arange(len(SESSION_LABELS))

# Individual animal lines
for animal in sorted(p2p_df['animal'].unique()):
    adf    = p2p_df[p2p_df['animal'] == animal]
    xs, ys = [], []
    for i, sl in enumerate(SESSION_LABELS):
        row = adf[adf['session'] == sl]
        if len(row) > 0:
            xs.append(i)
            ys.append(row['val'].values[0])
    if len(xs) > 1:
        ax.plot(xs, ys, 'o-', color='#7B52A6', alpha=0.4,
                markersize=8, linewidth=1.5)
    elif len(xs) == 1:
        ax.scatter(xs, ys, color='#7B52A6', alpha=0.4, s=64, zorder=3)

# Group mean ± SEM
means, sems = [], []
for sl in SESSION_LABELS:
    vals = p2p_df[p2p_df['session'] == sl]['val'].dropna()
    means.append(vals.mean() if len(vals) > 0 else np.nan)
    sems.append(vals.sem()   if len(vals) > 1 else 0)

ax.errorbar(x, means, yerr=sems,
            fmt='s-', color='black', markerfacecolor='black',
            markersize=12, linewidth=2.5,
            capsize=5, capthick=2, zorder=10,
            label='Group mean ± SEM')

if P2P_MODE == 'normalized':
    ax.axhline(1.0, color='gray', linewidth=1.0, linestyle='--', alpha=0.6)
    ax.axhline(0.0, color='gray', linewidth=1.0, linestyle=':', alpha=0.6)

ax.set_xticks(x)
ax.set_xticklabels(SESSION_XTICK_LABELS, fontsize=10)
ax.set_ylabel(ylabel, fontsize=11)
ax.set_title('Point-to-Plane Distance', fontsize=12)
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

for ext in ('png', 'svg'):
    out = os.path.join(OUTPUT_DIR, f'point_to_plane_social_{P2P_MODE}.{ext}')
    plt.savefig(out, dpi=200, bbox_inches='tight')
    print(f'Saved: {out}')

plt.show()


In [ ]:
# ── P2P Statistics ────────────────────────────────────────────────────────────
# 1. Pooled one-sample Wilcoxon against 0 (H1: non-planar)
# 2. One-sample Wilcoxon against 0 per session + BH (per-session descriptive)
# 3. Selected pairwise paired Wilcoxons across sessions + BH (stability claim)
#    Pairs: (1,2), (3,4), (1,3), (2,4), (4,5)

P2P_VALUE_COL = 'p2p_norm'

p2p_stats_records = []

# ── 1. Pooled one-sample Wilcoxon ────────────────────────────────────────────
pooled_vals = p2p_df[P2P_VALUE_COL].dropna().values
pool_stat, pool_p = stats.wilcoxon(pooled_vals, alternative='greater')
print(f'\n── P2P: Pooled one-sample Wilcoxon vs. 0 (n={len(pooled_vals)} animal-sessions) ─')
print(f'  mean={pooled_vals.mean():.4f}  median={np.median(pooled_vals):.4f}  sem={stats.sem(pooled_vals):.4f}')
print(f'  statistic={pool_stat:.1f}  p={pool_p:.2e}')

p2p_stats_records.append(pd.DataFrame([{
    'test':      'pooled',
    'session':   'ALL',
    'n':         len(pooled_vals),
    'mean':      pooled_vals.mean(),
    'sem':       stats.sem(pooled_vals),
    'median':    np.median(pooled_vals),
    'statistic': pool_stat,
    'p_value':   pool_p,
    'p_fdr':     np.nan,
    'sig':       pool_p < 0.05,
}]))

# ── 2. One-sample Wilcoxon per session + BH ───────────────────────────────────
onesample_rows = []
for session_label in SESSION_LABELS:
    vals = p2p_df[p2p_df['session'] == session_label][P2P_VALUE_COL].dropna().values
    if len(vals) < 3:
        print(f'  {session_label}: insufficient data (n={len(vals)}), skipping')
        continue
    stat, p = stats.wilcoxon(vals, alternative='greater')
    onesample_rows.append({
        'test':      'one_sample',
        'session':   session_label,
        'n':         len(vals),
        'mean':      vals.mean(),
        'sem':       stats.sem(vals),
        'median':    np.median(vals),
        'statistic': stat,
        'p_value':   p,
    })

onesample_df = pd.DataFrame(onesample_rows)
_, p_fdr, _, _ = multipletests(onesample_df['p_value'], alpha=0.05, method='fdr_bh')
onesample_df['p_fdr'] = p_fdr
onesample_df['sig']   = p_fdr < 0.05
p2p_stats_records.append(onesample_df)

print('\n── P2P: One-sample Wilcoxon vs. 0 per session (BH corrected) ───────────')
print(onesample_df[['session', 'n', 'mean', 'sem', 'median',
                     'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

# ── 3. Selected pairwise paired Wilcoxons + BH ───────────────────────────────
p2p_pivot = (p2p_df
             .pivot(index='animal', columns='session', values=P2P_VALUE_COL)
             .reindex(columns=SESSION_LABELS))

paired_rows = []
for sl_a, sl_b, comp_name in SELECTED_PAIRWISE_COMPARISONS:
    a = p2p_pivot[sl_a].dropna()
    b = p2p_pivot[sl_b].dropna()
    common = a.index.intersection(b.index)
    if len(common) < 3:
        print(f'  Skipping {comp_name}: insufficient paired data (n={len(common)})')
        continue
    stat, p = stats.wilcoxon(a.loc[common].values, b.loc[common].values,
                             alternative='two-sided')
    paired_rows.append({
        'test':       'selected_pairwise',
        'comparison': comp_name,
        'n':          len(common),
        'mean_a':     a.loc[common].mean(),
        'mean_b':     b.loc[common].mean(),
        'statistic':  stat,
        'p_value':    p,
    })

paired_df = pd.DataFrame(paired_rows)
_, p_fdr_paired, _, _ = multipletests(paired_df['p_value'], alpha=0.05, method='fdr_bh')
paired_df['p_fdr'] = p_fdr_paired
paired_df['sig']   = p_fdr_paired < 0.05
p2p_stats_records.append(paired_df)

print('\n── P2P: Selected pairwise Wilcoxons across sessions (BH corrected) ─────')
print(paired_df[['comparison', 'n', 'mean_a', 'mean_b',
                  'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
p2p_stats_csv = os.path.join(OUTPUT_DIR, 'p2p_normalized_statistics.csv')
pd.concat(p2p_stats_records, ignore_index=True).to_csv(p2p_stats_csv, index=False)
print(f'\nSaved: {p2p_stats_csv}')

## Parallelism

In [ ]:
def parallelism_score(v1, v2):
    v1_norm = v1 / np.linalg.norm(v1)
    v2_norm = v2 / np.linalg.norm(v2)
    return np.dot(v1_norm, v2_norm)

# ── Collect ───────────────────────────────────────────────────────────────────
ps_records = []

for session_label in SESSION_LABELS:
    pickle_path = os.path.join(PICKLE_DIR, f'{session_label}.pkl')
    if not os.path.isfile(pickle_path):
        continue
    with open(pickle_path, 'rb') as f:
        session_data = pickle.load(f)
    session_data = filter_session_data(session_data, session_label)
    for animal_key, animal_data in session_data.items():
        if not animal_data.get('included', False):
            continue
        animal_id = animal_key.split('_')[0]
        try:
            pt_pvs = animal_data['pt_pvs']
            c_pr = compute_condition_centroid(pt_pvs, 'pr')
            c_pl = compute_condition_centroid(pt_pvs, 'pl')
            c_nr = compute_condition_centroid(pt_pvs, 'nr')
            c_nl = compute_condition_centroid(pt_pvs, 'nl')

            id_ps  = parallelism_score(c_nl - c_pl, c_nr - c_pr)
            loc_ps = parallelism_score(c_pr - c_pl, c_nr - c_nl)

            ps_records.append({
                'animal':      animal_id,
                'session':     session_label,
                'identity_ps': id_ps,
                'location_ps': loc_ps,
            })
            print(f'{animal_key} | identity PS: {id_ps:.3f} | location PS: {loc_ps:.3f}')
        except Exception as e:
            print(f'  Skipped {animal_key}: {e}')

ps_df = pd.DataFrame(ps_records)

# ── CSV ───────────────────────────────────────────────────────────────────────
ps_csv_path = os.path.join(OUTPUT_DIR, 'parallelism_scores.csv')
ps_df.to_csv(ps_csv_path, index=False)
print(f'\nSaved CSV: {ps_csv_path}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, metric, color, title in zip(
    axes,
    ['identity_ps', 'location_ps'],
    [IDENTITY_COLOR, LOCATION_COLOR],
    ['Identity Parallelism Score', 'Location Parallelism Score'],
):
    x = np.arange(len(SESSION_LABELS))

    # Individual animal lines
    for animal in sorted(ps_df['animal'].unique()):
        adf    = ps_df[ps_df['animal'] == animal]
        xs, ys = [], []
        for i, sl in enumerate(SESSION_LABELS):
            row = adf[adf['session'] == sl]
            if len(row) > 0:
                xs.append(i)
                ys.append(row[metric].values[0])
        if len(xs) > 1:
            ax.plot(xs, ys, 'o-', color=color, alpha=0.4,
                    markersize=8, linewidth=1.5)
        elif len(xs) == 1:
            ax.scatter(xs, ys, color=color, alpha=0.4, s=64, zorder=3)

    # Group mean ± SEM
    means, sems = [], []
    for sl in SESSION_LABELS:
        vals = ps_df[ps_df['session'] == sl][metric].dropna()
        means.append(vals.mean() if len(vals) > 0 else np.nan)
        sems.append(vals.sem()   if len(vals) > 1 else 0)

    ax.errorbar(x, means, yerr=sems,
                fmt='s-', color='black', markerfacecolor='black',
                markersize=12, linewidth=2.5,
                capsize=5, capthick=2, zorder=10,
                label='Group mean ± SEM')

    ax.axhline(0,  color='gray',      linestyle='--', linewidth=0.8, alpha=0.8)
    ax.axhline(1,  color='lightgray', linestyle=':',  linewidth=0.8, alpha=0.8)
    ax.axhline(-1, color='lightgray', linestyle=':',  linewidth=0.8, alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(SESSION_XTICK_LABELS, fontsize=10)
    ax.set_ylabel('Parallelism score (cosine similarity)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(-1.1, 1.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Parallelism score across sessions\n'
             '(1 = parallel, 0 = orthogonal, −1 = anti-parallel)',
             fontsize=11)
plt.tight_layout()

for ext in ('png', 'svg'):
    out = os.path.join(OUTPUT_DIR, f'parallelism_scores.{ext}')
    plt.savefig(out, dpi=200, bbox_inches='tight')
    print(f'Saved: {out}')

plt.show()

print('\nSession means:')
print(ps_df.groupby('session')[['identity_ps', 'location_ps']]
          .mean().reindex(SESSION_LABELS).round(3).to_string())

In [ ]:
# Parallelism Statistics 
# 1. Pooled one-sample Wilcoxon against 0 (primary claim)
# 2. One-sample Wilcoxon against 0 per session + BH (per-session descriptive)
# 3. Selected pairwise paired Wilcoxons across sessions + BH (stability claim)
# 4. Pooled identity vs location paired Wilcoxon
# 5. Identity vs location per session + BH

ps_stats_records = []

for ps_col, axis_label in [('identity_ps', 'Identity'), ('location_ps', 'Location')]:

    print(f'\n{"="*60}')
    print(f'PARALLELISM — {axis_label} axis')

    # ── 1. Pooled one-sample Wilcoxon ────────────────────────────────────────
    pooled_vals = ps_df[ps_col].dropna().values
    pool_stat, pool_p = stats.wilcoxon(pooled_vals, alternative='two-sided')
    print(f'\n── Pooled one-sample Wilcoxon vs. 0 (n={len(pooled_vals)} animal-sessions) ─')
    print(f'  mean={pooled_vals.mean():.4f}  median={np.median(pooled_vals):.4f}  '
          f'sem={stats.sem(pooled_vals):.4f}')
    print(f'  statistic={pool_stat:.1f}  p={pool_p:.2e}')
    ps_stats_records.append(pd.DataFrame([{
        'axis':      axis_label,
        'test':      'pooled',
        'session':   'ALL',
        'n':         len(pooled_vals),
        'mean':      pooled_vals.mean(),
        'sem':       stats.sem(pooled_vals),
        'median':    np.median(pooled_vals),
        'statistic': pool_stat,
        'p_value':   pool_p,
        'p_fdr':     np.nan,
        'sig':       pool_p < 0.05,
    }]))

    # ── 2. One-sample Wilcoxon per session + BH ───────────────────────────────
    onesample_rows = []
    for session_label in SESSION_LABELS:
        vals = ps_df[ps_df['session'] == session_label][ps_col].dropna().values
        if len(vals) < 3:
            print(f'  {session_label}: insufficient data (n={len(vals)}), skipping')
            continue
        stat, p = stats.wilcoxon(vals, alternative='two-sided')
        onesample_rows.append({
            'axis':      axis_label,
            'session':   session_label,
            'n':         len(vals),
            'mean':      vals.mean(),
            'sem':       stats.sem(vals),
            'median':    np.median(vals),
            'statistic': stat,
            'p_value':   p,
        })

    os_df = pd.DataFrame(onesample_rows)
    _, p_fdr, _, _ = multipletests(os_df['p_value'], alpha=0.05, method='fdr_bh')
    os_df['p_fdr'] = p_fdr
    os_df['sig']   = p_fdr < 0.05
    ps_stats_records.append(os_df.assign(test='one_sample'))

    print(f'\n── Per-session Wilcoxon vs. 0 (BH corrected) ───────────────────────')
    print(os_df[['session', 'n', 'mean', 'sem', 'median',
                  'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

    # ── 3. Selected pairwise paired Wilcoxons + BH ───────────────────────────
    ps_pivot = (ps_df
                .pivot(index='animal', columns='session', values=ps_col)
                .reindex(columns=SESSION_LABELS))

    paired_rows = []
    for sl_a, sl_b, comp_name in SELECTED_PAIRWISE_COMPARISONS:
        a = ps_pivot[sl_a].dropna()
        b = ps_pivot[sl_b].dropna()
        common = a.index.intersection(b.index)
        if len(common) < 3:
            print(f'  Skipping {comp_name}: insufficient paired data (n={len(common)})')
            continue
        stat, p = stats.wilcoxon(a.loc[common].values, b.loc[common].values,
                                 alternative='two-sided')
        paired_rows.append({
            'axis':       axis_label,
            'comparison': comp_name,
            'n':          len(common),
            'mean_a':     a.loc[common].mean(),
            'mean_b':     b.loc[common].mean(),
            'statistic':  stat,
            'p_value':    p,
        })

    paired_df = pd.DataFrame(paired_rows)
    if len(paired_df) > 0:
        _, p_fdr_paired, _, _ = multipletests(paired_df['p_value'], alpha=0.05, method='fdr_bh')
        paired_df['p_fdr'] = p_fdr_paired
        paired_df['sig']   = p_fdr_paired < 0.05
        ps_stats_records.append(paired_df.assign(test='selected_pairwise'))

        print(f'\n── Selected pairwise Wilcoxons across sessions (BH corrected) ───────')
        print(paired_df[['comparison', 'n', 'mean_a', 'mean_b',
                           'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

# ── 4. Pooled identity vs location paired Wilcoxon ───────────────────────────
print(f'\n{"="*60}')
print('PARALLELISM — IDENTITY vs LOCATION')

id_pooled  = ps_df[['animal', 'session', 'identity_ps']].dropna()
loc_pooled = ps_df[['animal', 'session', 'location_ps']].dropna()
merged     = pd.merge(id_pooled, loc_pooled, on=['animal', 'session'])

pool_stat4, pool_p4 = stats.wilcoxon(merged['identity_ps'].values,
                                      merged['location_ps'].values,
                                      alternative='two-sided')
print(f'\n── Pooled identity vs location (n={len(merged)} animal-sessions) ──────')
print(f'  mean_identity={merged["identity_ps"].mean():.4f}  '
      f'mean_location={merged["location_ps"].mean():.4f}')
print(f'  statistic={pool_stat4:.1f}  p={pool_p4:.2e}')

ps_stats_records.append(pd.DataFrame([{
    'axis':      'identity_vs_location',
    'test':      'pooled_id_vs_loc',
    'session':   'ALL',
    'n':         len(merged),
    'mean':      np.nan,
    'sem':       np.nan,
    'median':    np.nan,
    'statistic': pool_stat4,
    'p_value':   pool_p4,
    'p_fdr':     np.nan,
    'sig':       pool_p4 < 0.05,
}]))

# ── 5. Identity vs location per session + BH ─────────────────────────────────
t5_rows = []
for session_label in SESSION_LABELS:
    id_vals  = ps_df[ps_df['session'] == session_label][['animal', 'identity_ps']].dropna().set_index('animal')
    loc_vals = ps_df[ps_df['session'] == session_label][['animal', 'location_ps']].dropna().set_index('animal')
    common   = id_vals.index.intersection(loc_vals.index)
    if len(common) < 3:
        print(f'  {session_label}: insufficient paired data (n={len(common)}), skipping')
        continue
    stat, p = stats.wilcoxon(id_vals.loc[common, 'identity_ps'].values,
                             loc_vals.loc[common, 'location_ps'].values,
                             alternative='two-sided')
    t5_rows.append({
        'axis':           'identity_vs_location',
        'test':           'per_session_id_vs_loc',
        'session':        session_label,
        'n':              len(common),
        'mean_identity':  id_vals.loc[common, 'identity_ps'].mean(),
        'mean_location':  loc_vals.loc[common, 'location_ps'].mean(),
        'statistic':      stat,
        'p_value':        p,
    })

t5_df = pd.DataFrame(t5_rows)
_, p_fdr_t5, _, _ = multipletests(t5_df['p_value'], alpha=0.05, method='fdr_bh')
t5_df['p_fdr'] = p_fdr_t5
t5_df['sig']   = p_fdr_t5 < 0.05
ps_stats_records.append(t5_df)

print(f'\n── Identity vs location per session (BH corrected) ─────────────────')
print(t5_df[['session', 'n', 'mean_identity', 'mean_location',
              'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
if ps_stats_records:
    all_ps_stats = pd.concat(ps_stats_records, ignore_index=True)
    ps_stats_csv = os.path.join(OUTPUT_DIR, 'parallelism_statistics.csv')
    all_ps_stats.to_csv(ps_stats_csv, index=False)
    print(f'\nSaved: {ps_stats_csv}')

In [ ]:
# Parallelism LMM: Identity vs Location

# Reshape to long format
ps_long = pd.melt(
    ps_df,
    id_vars=['animal', 'session'],
    value_vars=['identity_ps', 'location_ps'],
    var_name='axis',
    value_name='parallelism_score'
)
ps_long['axis'] = ps_long['axis'].map({'identity_ps': 'identity', 'location_ps': 'location'})
ps_long = ps_long.dropna(subset=['parallelism_score'])
ps_long['axis'] = pd.Categorical(ps_long['axis'], categories=['location', 'identity'])
ps_long = ps_long.rename(columns={'session': 'session_label'})

print('\n' + '='*60)
print('PARALLELISM — IDENTITY vs LOCATION')

ps_results = run_lmm(ps_long, 'parallelism_score', 'axis', 'session_label', 'animal')

# Save
lmm_csv = os.path.join(OUTPUT_DIR, 'parallelism_lmm_results.csv')
pd.DataFrame([{'comparison': 'identity_vs_location', **ps_results}]).to_csv(lmm_csv, index=False)
print(f'\nSaved: {lmm_csv}')

## CCGP

In [ ]:
# Load
all_results = {}   # {session_label: {animal_key: ar}}

for fname in sorted(os.listdir(RESULTS_DIR)):
    if not fname.endswith('_ccgp.pkl'):
        continue
    session_label = next(
        (sl for sl in SESSION_LABELS if fname.replace('_ccgp.pkl', '').endswith(sl)),
        None
    )
    if session_label is None:
        continue
    animal_key = fname.replace('_ccgp.pkl', '')
    if is_excluded(animal_key, session_label):
        continue
    with open(os.path.join(RESULTS_DIR, fname), 'rb') as f:
        ar = pickle.load(f)
    all_results.setdefault(session_label, {})[animal_key] = ar

# Build CSV
null_key_map = {'accuracy': 'null_accuracies', 'f1': 'null_f1s', 'auroc': 'null_aurocs'}
ccgp_records = []

for session_label in SESSION_LABELS:
    if session_label not in all_results:
        continue
    for animal_key, ar in all_results[session_label].items():
        animal_id   = animal_key.split('_')[0]
        session_idx = SESSION_LABELS.index(session_label) + 1
        for decode_type, full_key, within_key in [
            ('identity', 'identity_full', 'identity_within_animal'),
            ('location', 'location_full', 'location_within_animal'),
        ]:
            if full_key not in ar:
                continue
            full_res  = ar[full_key]
            true_full = full_res['true_decoding_results']
            null      = full_res['null_condition_shuffle_results']

            # within-animal subsampled true (may not exist if animal below threshold)
            within_res  = ar.get(within_key, None)
            true_within = within_res['true_decoding_results'] if within_res is not None else None

            ccgp_records.append({
                'animal':                    animal_id,
                'session_idx':               session_idx,
                'session_label':             session_label,
                'decode_type':               decode_type,
                'metric':                    METRIC,
                # full-population true
                'true_full_value':           true_full[f'mean_{METRIC}'],
                'true_full_std':             true_full[f'std_{METRIC}'],
                'n_neurons_full':            full_res['n_neurons_subsampled'],
                # within-animal subsampled true
                'true_within_value':         true_within[f'mean_{METRIC}'] if true_within is not None else np.nan,
                'true_within_std':           true_within[f'std_{METRIC}']  if true_within is not None else np.nan,
                'n_neurons_within':          within_res['n_neurons_subsampled'] if within_res is not None else np.nan,
                # null (from full population)
                'null_mean':                 null[f'mean_null_{METRIC}'],
                'null_std':                  null[f'std_null_{METRIC}'],
                'null_values':               str(null[null_key_map[METRIC]]),
                'p_value':                   full_res[f'p_val_condition_shuffle_{METRIC}'],
            })

ccgp_df = pd.DataFrame(ccgp_records)
csv_path = os.path.join(OUTPUT_DIR, f'ccgp_{METRIC}.csv')
ccgp_df.to_csv(csv_path, index=False)
print(f'Saved CSV: {csv_path}')

# Plot (within-animal subsampled trues)
metric_label = {'accuracy': 'Accuracy', 'f1': 'F1 Score', 'auroc': 'AUROC'}[METRIC]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, decode_type, color, title in zip(
    axes,
    ['identity', 'location'],
    [IDENTITY_COLOR, LOCATION_COLOR],
    ['Identity CCGP', 'Location CCGP'],
):
    sub = ccgp_df[ccgp_df['decode_type'] == decode_type]
    x   = np.arange(len(SESSION_LABELS))

    # Individual animal lines (within-animal subsampled)
    for animal in sorted(sub['animal'].unique()):
        adf    = sub[sub['animal'] == animal]
        xs, ys = [], []
        for i, sl in enumerate(SESSION_LABELS):
            row = adf[adf['session_label'] == sl]
            if len(row) > 0 and not np.isnan(row['true_within_value'].values[0]):
                xs.append(i)
                ys.append(row['true_within_value'].values[0])
        if len(xs) > 1:
            ax.plot(xs, ys, 'o-', color=color, alpha=0.4,
                    markersize=8, linewidth=1.5)
        elif len(xs) == 1:
            ax.scatter(xs, ys, color=color, alpha=0.4, s=64, zorder=3)

    # Group mean ± SEM (within-animal subsampled)
    means, sems = [], []
    for sl in SESSION_LABELS:
        vals = sub[sub['session_label'] == sl]['true_within_value'].dropna()
        means.append(vals.mean() if len(vals) > 0 else np.nan)
        sems.append(vals.sem()   if len(vals) > 1 else 0)

    ax.errorbar(x, means, yerr=sems,
                fmt='s-', color='black', markerfacecolor='black',
                markersize=12, linewidth=2.5,
                capsize=5, capthick=2, zorder=10,
                label='Group mean ± SEM')

    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(SESSION_XTICK_LABELS, fontsize=10)
    ax.set_ylabel(metric_label, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(0.0, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'CCGP {metric_label}: individual animals + group mean ± SEM\n'
             f'(per-animal subsampled to session minimum)',
             fontsize=11)
plt.tight_layout()

for ext in ('png', 'svg'):
    out = os.path.join(OUTPUT_DIR, f'ccgp_{METRIC}.{ext}')
    plt.savefig(out, dpi=200, bbox_inches='tight')
    print(f'Saved: {out}')

plt.show()

print('\nCCGP means per session:')
print(ccgp_df.groupby(['decode_type', 'session_label'])['true_within_value']
          .mean().reindex(pd.MultiIndex.from_product(
              [['identity', 'location'], SESSION_LABELS],
              names=['decode_type', 'session_label']))
          .round(3).to_string())

## Statistics

In [ ]:
# CCGP Statistics
# Test 1: Pooled Wilcoxon (full-population true vs null mean, all animal-sessions)
# Test 2: Per-session Wilcoxon (full-population true vs null mean, BH across 5 sessions)
# Test 3: Selected pairwise cross-session Wilcoxons on within-animal subsampled trues
# Test 4: Identity vs location within each session on full population trues, BH across 5 sessions
# Test 5: Pooled identity vs location paired Wilcoxon

all_stats_records = []

for decode_type in ['identity', 'location']:
    sub = ccgp_df[ccgp_df['decode_type'] == decode_type].copy()

    print(f'\n{"="*60}')
    print(f'CCGP — {decode_type.upper()}')

    # ── Test 1: Pooled Wilcoxon (full true vs null mean, all animal-sessions) ─
    true_all = sub['true_full_value'].dropna()
    null_all = sub.loc[true_all.index, 'null_mean']
    pool_stat, pool_p = stats.wilcoxon(true_all.values, null_all.values, alternative='greater')

    print(f'\n── Test 1: Pooled Wilcoxon (full true vs null mean, n={len(true_all)} animal-sessions) ─')
    print(f'  mean_true={true_all.mean():.4f}  mean_null={null_all.mean():.4f}')
    print(f'  statistic={pool_stat:.1f}  p={pool_p:.2e}')

    t1_df = pd.DataFrame([{
        'decode_type': decode_type,
        'test':        'pooled_vs_null',
        'n':           len(true_all),
        'mean_true':   true_all.mean(),
        'mean_null':   null_all.mean(),
        'statistic':   pool_stat,
        'p_value':     pool_p,
    }])
    all_stats_records.append(t1_df)

    # ── Test 2: Per-session Wilcoxon (full true vs null mean) + BH ───────────
    t2_rows = []
    for session_label in SESSION_LABELS:
        sess = sub[sub['session_label'] == session_label].dropna(
            subset=['true_full_value', 'null_mean'])
        if len(sess) < 3:
            print(f'  {session_label}: insufficient data (n={len(sess)}), skipping')
            continue
        true_vals = sess['true_full_value'].values
        null_vals = sess['null_mean'].values
        stat, p = stats.wilcoxon(true_vals, null_vals, alternative='greater')
        t2_rows.append({
            'decode_type': decode_type,
            'session':     session_label,
            'n':           len(sess),
            'mean_true':   true_vals.mean(),
            'mean_null':   null_vals.mean(),
            'statistic':   stat,
            'p_value':     p,
        })

    t2_df = pd.DataFrame(t2_rows)
    _, p_fdr_t2, _, _ = multipletests(t2_df['p_value'], alpha=0.05, method='fdr_bh')
    t2_df['p_fdr'] = p_fdr_t2
    t2_df['sig']   = p_fdr_t2 < 0.05
    t2_df['test']  = 'per_session_vs_null'
    all_stats_records.append(t2_df)

    print(f'\n── Test 2: Per-session Wilcoxon (full true vs null mean, BH corrected) ─')
    print(f'  Note: underpowered given n=4-6 per session')
    print(t2_df[['session', 'n', 'mean_true', 'mean_null',
                  'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

    # ── Test 3: Selected pairwise cross-session Wilcoxons + BH ───────────────
    t3_rows = []
    for sl_a, sl_b, comp_name in SELECTED_PAIRWISE_COMPARISONS:
        a = sub[sub['session_label'] == sl_a][['animal', 'true_within_value']].set_index('animal')
        b = sub[sub['session_label'] == sl_b][['animal', 'true_within_value']].set_index('animal')
        common = a.index.intersection(b.index)
        if len(common) < 3:
            print(f'  Skipping {comp_name}: insufficient paired data (n={len(common)})')
            continue
        a_vals = a.loc[common, 'true_within_value'].dropna()
        b_vals = b.loc[common, 'true_within_value'].dropna()
        common = a_vals.index.intersection(b_vals.index)
        if len(common) < 3:
            print(f'  Skipping {comp_name}: insufficient non-NaN paired data (n={len(common)})')
            continue
        stat, p = stats.wilcoxon(a_vals.loc[common].values,
                                 b_vals.loc[common].values,
                                 alternative='two-sided')
        t3_rows.append({
            'decode_type':     decode_type,
            'comparison':      comp_name,
            'n':               len(common),
            'mean_a':          a_vals.loc[common].mean(),
            'mean_b':          b_vals.loc[common].mean(),
            'difference_mean': (b_vals.loc[common] - a_vals.loc[common]).mean(),
            'statistic':       stat,
            'p_value':         p,
        })

    t3_df = pd.DataFrame(t3_rows)
    _, p_fdr_t3, _, _ = multipletests(t3_df['p_value'], alpha=0.05, method='fdr_bh')
    t3_df['p_fdr'] = p_fdr_t3
    t3_df['sig']   = p_fdr_t3 < 0.05
    t3_df['test']  = 'cross_session_paired'
    all_stats_records.append(t3_df)

    print(f'\n── Test 3: Selected pairwise cross-session Wilcoxons (BH corrected) ─')
    print(t3_df[['comparison', 'n', 'mean_a', 'mean_b', 'difference_mean',
                  'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

# ── Test 4: Identity vs location within each session + BH ────────────────────
print(f'\n{"="*60}')
print('CCGP — IDENTITY vs LOCATION (within session)')

t4_rows = []
for session_label in SESSION_LABELS:
    id_vals  = ccgp_df[(ccgp_df['decode_type'] == 'identity') &
                        (ccgp_df['session_label'] == session_label)][['animal', 'true_full_value']].set_index('animal')
    loc_vals = ccgp_df[(ccgp_df['decode_type'] == 'location') &
                        (ccgp_df['session_label'] == session_label)][['animal', 'true_full_value']].set_index('animal')
    common = id_vals.index.intersection(loc_vals.index)
    if len(common) < 3:
        print(f'  {session_label}: insufficient paired data (n={len(common)}), skipping')
        continue
    id_vals_clean  = id_vals.loc[common, 'true_full_value'].dropna()
    loc_vals_clean = loc_vals.loc[common, 'true_full_value'].dropna()
    common = id_vals_clean.index.intersection(loc_vals_clean.index)
    if len(common) < 3:
        print(f'  {session_label}: insufficient non-NaN paired data (n={len(common)}), skipping')
        continue
    stat, p = stats.wilcoxon(id_vals_clean.loc[common].values,
                             loc_vals_clean.loc[common].values,
                             alternative='two-sided')
    t4_rows.append({
        'session':        session_label,
        'n':              len(common),
        'mean_identity':  id_vals_clean.loc[common].mean(),
        'mean_location':  loc_vals_clean.loc[common].mean(),
        'statistic':      stat,
        'p_value':        p,
    })

t4_df = pd.DataFrame(t4_rows)
_, p_fdr_t4, _, _ = multipletests(t4_df['p_value'], alpha=0.05, method='fdr_bh')
t4_df['p_fdr'] = p_fdr_t4
t4_df['sig']   = p_fdr_t4 < 0.05
t4_df['test']  = 'identity_vs_location'
all_stats_records.append(t4_df)

print(f'\n── Test 4: Identity vs location within session (BH corrected) ───────')
print(t4_df[['session', 'n', 'mean_identity', 'mean_location',
              'statistic', 'p_value', 'p_fdr', 'sig']].round(4).to_string(index=False))

# ── Test 5: Pooled identity vs location paired Wilcoxon ──────────────────────
print(f'\n{"="*60}')
print('CCGP — IDENTITY vs LOCATION (pooled)')

id_pooled  = ccgp_df[ccgp_df['decode_type'] == 'identity'][['animal', 'session_label', 'true_full_value']].dropna()
loc_pooled = ccgp_df[ccgp_df['decode_type'] == 'location'][['animal', 'session_label', 'true_full_value']].dropna()
merged5    = pd.merge(id_pooled, loc_pooled, on=['animal', 'session_label'],
                      suffixes=('_identity', '_location'))

pool_stat5, pool_p5 = stats.wilcoxon(merged5['true_full_value_identity'].values,
                                      merged5['true_full_value_location'].values,
                                      alternative='two-sided')
print(f'\n── Pooled identity vs location (n={len(merged5)} animal-sessions) ──────')
print(f'  mean_identity={merged5["true_full_value_identity"].mean():.4f}  '
      f'mean_location={merged5["true_full_value_location"].mean():.4f}')
print(f'  statistic={pool_stat5:.1f}  p={pool_p5:.2e}')

all_stats_records.append(pd.DataFrame([{
    'decode_type': 'identity_vs_location',
    'test':        'pooled_id_vs_loc',
    'session':     'ALL',
    'n':           len(merged5),
    'mean_identity': merged5['true_full_value_identity'].mean(),
    'mean_location': merged5['true_full_value_location'].mean(),
    'statistic':   pool_stat5,
    'p_value':     pool_p5,
}]))

# ── Save ──────────────────────────────────────────────────────────────────────
all_stats_df = pd.concat(all_stats_records, ignore_index=True)
stats_csv    = os.path.join(OUTPUT_DIR, f'ccgp_{METRIC}_statistics.csv')
all_stats_df.to_csv(stats_csv, index=False)
print(f'\nSaved: {stats_csv}')

In [ ]:
# ── True vs Null: identity ────────────────────────────────────────────────────
print('\n' + '='*60)
print('TRUE vs NULL — IDENTITY')

sub_id = ccgp_df[ccgp_df['decode_type'] == 'identity'].copy()
true_id  = sub_id[['animal', 'session_label', 'true_within_value']].dropna().rename(
    columns={'true_within_value': 'accuracy'})
true_id['source'] = 'true'
null_id  = sub_id[['animal', 'session_label', 'null_mean']].dropna().rename(
    columns={'null_mean': 'accuracy'})
null_id['source'] = 'null'
lmm_id = pd.concat([true_id, null_id], ignore_index=True)
lmm_id['source'] = pd.Categorical(lmm_id['source'], categories=['null', 'true'])

id_tvn_results = run_lmm(lmm_id, 'accuracy', 'source', 'session_label', 'animal')

# ── True vs Null: location ────────────────────────────────────────────────────
print('\n' + '='*60)
print('TRUE vs NULL — LOCATION')

sub_loc = ccgp_df[ccgp_df['decode_type'] == 'location'].copy()
true_loc = sub_loc[['animal', 'session_label', 'true_within_value']].dropna().rename(
    columns={'true_within_value': 'accuracy'})
true_loc['source'] = 'true'
null_loc = sub_loc[['animal', 'session_label', 'null_mean']].dropna().rename(
    columns={'null_mean': 'accuracy'})
null_loc['source'] = 'null'
lmm_loc = pd.concat([true_loc, null_loc], ignore_index=True)
lmm_loc['source'] = pd.Categorical(lmm_loc['source'], categories=['null', 'true'])

loc_tvn_results = run_lmm(lmm_loc, 'accuracy', 'source', 'session_label', 'animal')

# ── Identity vs Location ──────────────────────────────────────────────────────
print('\n' + '='*60)
print('IDENTITY vs LOCATION')

ccgp_long = ccgp_df[['animal', 'session_label', 'decode_type', 'true_within_value']].dropna()
ccgp_long = ccgp_long.rename(columns={'true_within_value': 'accuracy'})
ccgp_long['decode_type'] = pd.Categorical(ccgp_long['decode_type'],
                                           categories=['location', 'identity'])

idvloc_results = run_lmm(ccgp_long, 'accuracy', 'decode_type', 'session_label', 'animal')

# ── Save all results ──────────────────────────────────────────────────────────
all_lmm = pd.DataFrame([
    {'comparison': 'true_vs_null_identity', **id_tvn_results},
    {'comparison': 'true_vs_null_location', **loc_tvn_results},
    {'comparison': 'identity_vs_location',  **idvloc_results},
])
lmm_csv = os.path.join(OUTPUT_DIR, f'ccgp_{METRIC}_lmm_results.csv')
all_lmm.to_csv(lmm_csv, index=False)
print(f'\nSaved: {lmm_csv}')

## Individual PCA

In [ ]:
def total_pseudotrials(animal_data):
    """Sum pseudotrials across all four conditions."""
    pt_pvs = animal_data['pt_pvs']
    return sum(len(pt_pvs[c]) for c in CONDITIONS if c in pt_pvs)

def compute_pseudotrial_coords(animal_data, pca):
    """Project individual pseudotrials for each condition into PCA space."""
    pt_pvs = animal_data['pt_pvs']
    pseudo_coords = {}
    for cond in CONDITIONS:
        if cond not in pt_pvs or len(pt_pvs[cond]) == 0:
            pseudo_coords[cond] = np.empty((0, 3))
            continue
        matrix = np.stack([v.values for v in pt_pvs[cond].values()])
        pseudo_coords[cond] = pca.transform(matrix)
    return pseudo_coords

# ── Select top 3 animals globally by mean pseudotrials across sessions ────────
animal_pt_totals = {}   # {animal_id: {session_label: n_pts}}

for session_label in SESSION_LABELS:
    pickle_path = os.path.join(PICKLE_DIR, f'{session_label}.pkl')
    if not os.path.isfile(pickle_path):
        continue
    with open(pickle_path, 'rb') as f:
        session_data = pickle.load(f)
    session_data = filter_session_data(session_data, session_label)
    for animal_key, animal_data in session_data.items():
        if not animal_data.get('included', False):
            continue
        animal_id = animal_key.split('_')[0]
        n_pts = total_pseudotrials(animal_data)
        animal_pt_totals.setdefault(animal_id, {})[session_label] = n_pts
    del session_data

# Mean across sessions each animal appears in
animal_means = {
    aid: np.mean(list(sessions.values()))
    for aid, sessions in animal_pt_totals.items()
}
top_3_animals = sorted(animal_means, key=animal_means.get, reverse=True)[:3]

print('Top 3 animals by mean pseudotrials across sessions:')
for aid in top_3_animals:
    session_counts = animal_pt_totals[aid]
    mean_pts = animal_means[aid]
    print(f'  {aid}: mean={mean_pts:.0f}  |  per session: {session_counts}')

# ── Figure: 3 rows × 5 columns ───────────────────────────────────────────────
fig = plt.figure(figsize=(5 * 2, 3 * 4))

# Need a lookup: animal_id → animal_key per session
animal_key_lookup = {}   # {session_label: {animal_id: animal_key}}

for session_label in SESSION_LABELS:
    pickle_path = os.path.join(PICKLE_DIR, f'{session_label}.pkl')
    if not os.path.isfile(pickle_path):
        animal_key_lookup[session_label] = {}
        continue
    with open(pickle_path, 'rb') as f:
        session_data = pickle.load(f)
    session_data = filter_session_data(session_data, session_label)
    animal_key_lookup[session_label] = {
        k.split('_')[0]: k
        for k, v in session_data.items() if v.get('included', False)
    }
    del session_data

for row, animal_id in enumerate(top_3_animals):
    for col, session_label in enumerate(SESSION_LABELS):
        ax = fig.add_subplot(3, 5, row * 5 + col + 1, projection='3d')

        # Pane styling — applied regardless so grayed cells look consistent
        ax.xaxis.pane.fill = True
        ax.yaxis.pane.fill = True
        ax.zaxis.pane.fill = True
        ax.xaxis.pane.set_edgecolor('#BBBBBB')
        ax.yaxis.pane.set_edgecolor('#BBBBBB')
        ax.zaxis.pane.set_edgecolor('#BBBBBB')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.grid(False)
        ax.set_axis_on()

        # Check if this animal has data for this session
        if animal_id not in animal_key_lookup.get(session_label, {}):
            # Gray out
            ax.xaxis.pane.set_facecolor('#D0D0D0')
            ax.yaxis.pane.set_facecolor('#D0D0D0')
            ax.zaxis.pane.set_facecolor('#D0D0D0')
            subtitle = 'no data'
            if row == 0:
                subtitle = f'{SESSION_XTICK_LABELS[col]}\n{subtitle}'
            ax.set_title(subtitle, fontsize=7, pad=6, color='gray')
            continue

        # Load this animal's data
        pickle_path = os.path.join(PICKLE_DIR, f'{session_label}.pkl')
        with open(pickle_path, 'rb') as f:
            session_data = pickle.load(f)
        animal_key  = animal_key_lookup[session_label][animal_id]
        animal_data = session_data[animal_key]
        n_pts       = total_pseudotrials(animal_data)

        try:
            centroids             = build_animal_centroids(animal_data)
            coords, pca, var_exp  = run_pca_on_centroids(centroids)
            coord_dict            = {cond: coords[i] for i, cond in enumerate(CONDITIONS)}
            pseudo_coords         = compute_pseudotrial_coords(animal_data, pca)

            # Normal pane colors
            ax.xaxis.pane.set_facecolor('#E8E8E8')
            ax.yaxis.pane.set_facecolor('#D8D8D8')
            ax.zaxis.pane.set_facecolor('#C8C8C8')

            # Individual pseudotrial dots
            for cond in CONDITIONS:
                pts = pseudo_coords[cond]
                if len(pts) == 0:
                    continue
                ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
                           color=CONDITION_COLORS[cond],
                           s=10, alpha=0.3, zorder=2,
                           edgecolors='none', depthshade=False)

            # Edges
            for c1, c2 in EDGES:
                p1, p2 = coord_dict[c1], coord_dict[c2]
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                        color='gray', linewidth=1.0, alpha=0.5, zorder=3)

            # Centroids
            for cond in CONDITIONS:
                x, y, z = coord_dict[cond]
                ax.scatter(x, y, z,
                           color=CONDITION_COLORS[cond], s=80, zorder=5,
                           edgecolors='black', linewidths=1.2,
                           depthshade=False)
                ax.text(x, y, z, f'  {cond}', fontsize=7,
                        color=CONDITION_COLORS[cond], fontweight='bold')

            ax.set_xlabel(f'PC1 ({var_exp[0]*100:.0f}%)', fontsize=6, labelpad=4)
            ax.set_ylabel(f'PC2 ({var_exp[1]*100:.0f}%)', fontsize=6, labelpad=4)
            ax.set_zlabel(f'PC3 ({var_exp[2]*100:.0f}%)', fontsize=6, labelpad=4)

            subtitle = f'Animal {animal_id} | {var_exp[0]*100:.0f}/{var_exp[1]*100:.0f}/{var_exp[2]*100:.0f}% (PC1/2/3)'
            if row == 0:
                subtitle = f'{SESSION_XTICK_LABELS[col]}\n{subtitle}'
            ax.set_title(subtitle, fontsize=7, pad=6)

        except Exception as e:
            print(f'  ERROR {animal_key}: {e}')

        del session_data

plt.suptitle('Per-animal PCA geometry (top 3 animals by mean pseudotrials across sessions)',
             fontsize=11, y=1.01)
plt.tight_layout()

for ext in ('png', 'svg'):
    out = os.path.join(OUTPUT_DIR, f'supplement_per_animal_pca.{ext}')
    plt.savefig(out, dpi=200, bbox_inches='tight')
    print(f'Saved: {out}')

plt.show()

In [ ]:
# Supplementary: per-session paired true vs. null plots
n_sessions = len(SESSION_LABELS)
x_null, x_true = 1, 0

for decode_type, fig_color, fig_title in [
    ('identity', IDENTITY_COLOR, 'Identity CCGP'),
    ('location', LOCATION_COLOR, 'Location CCGP'),
]:
    sub = ccgp_df[ccgp_df['decode_type'] == decode_type]

    fig, axes = plt.subplots(1, n_sessions, figsize=(3 * n_sessions, 5), sharey=True)
    if n_sessions == 1:
        axes = [axes]

    for ax, session_label, xtick_label in zip(axes, SESSION_LABELS, SESSION_XTICK_LABELS):
        sdf = sub[sub['session_label'] == session_label]

        for _, row in sdf.iterrows():
            true_val = row['true_full_value']
            null_val = row['null_mean']
            ax.plot([x_null, x_true], [null_val, true_val],
                    color=fig_color, alpha=0.5, linewidth=1.5)
            ax.scatter(x_null, null_val,
                       color='gray', s=64, zorder=3, alpha=0.7)
            ax.scatter(x_true, true_val,
                       color=fig_color, s=64, zorder=3, alpha=0.85)

        if len(sdf) > 0:
            true_mean, true_sem = sdf['true_full_value'].mean(), sdf['true_full_value'].sem()
            null_mean, null_sem = sdf['null_mean'].mean(), sdf['null_mean'].sem()
            ax.errorbar(x_true, true_mean, yerr=true_sem,
                        fmt='none', color='black',
                        capsize=6, capthick=2, elinewidth=2, zorder=5)
            ax.errorbar(x_null, null_mean, yerr=null_sem,
                        fmt='none', color='black',
                        capsize=6, capthick=2, elinewidth=2, zorder=5)

        ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, linewidth=1)
        ax.set_xticks([x_null, x_true])
        ax.set_xticklabels(['Null', 'True'], fontsize=10)
        ax.set_xlim(-0.4, 1.4)
        ax.set_ylim(0.0, 1.0)
        ax.set_title(xtick_label, fontsize=11)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[0].set_ylabel(metric_label, fontsize=11)
    fig.suptitle(
        f'Supplementary: {fig_title} — true vs. null per session\n'
        f'(full population, condition shuffle null)',
        fontsize=11
    )
    plt.tight_layout()

    for ext in ('png', 'svg'):
        out = os.path.join(OUTPUT_DIR, f'ccgp_{METRIC}_{decode_type}_supp_paired.{ext}')
        plt.savefig(out, dpi=200, bbox_inches='tight')
        print(f'Saved: {out}')

    plt.show()

# Save supplementary CSV (outside loop — contains both decode types)
supp_cols = ['animal', 'session_idx', 'session_label', 'decode_type',
             'metric', 'true_full_value', 'true_full_std', 'null_mean', 'null_std', 'p_value']
supp_csv_path = os.path.join(OUTPUT_DIR, f'ccgp_{METRIC}_supp_paired.csv')
ccgp_df[supp_cols].to_csv(supp_csv_path, index=False)
print(f'Saved supplementary CSV: {supp_csv_path}')